In [5]:
import pandas as pd
import numpy as np

import sys
from pathlib import Path
PROJECT_ROOT = Path().resolve()
while not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.append(str(PROJECT_ROOT))

from src.labeling import add_returns
from src.features import add_lagged_returns, add_moving_average_features
from src.volatility import add_close_to_close_volatility

df = pd.read_csv("../data/raw/nifty50.csv", parse_dates=["Date"])
df = df.sort_values("Date").reset_index(drop=True)

df = add_returns(df)

df = add_lagged_returns(df)
df = add_moving_average_features(df)

df = add_close_to_close_volatility(df)
df["vol_cc_lag_1"] = df["vol_cc"].shift(1)
df = df.dropna().reset_index(drop=True)
df.tail()

,Date,Open,High,Low,Close,Volume,return,ret_lag_1,ret_lag_5,ret_lag_10,ma_ratio_5,ma_ratio_10,ma_ratio_20,vol_cc,vol_cc_lag_1
4495,2026-02-13,25571.150391,25630.349609,25444.300781,25471.099609,453500,-0.013023,-0.005650,0.001985,-0.003865,0.986987,0.991236,0.998418,0.138191,0.130152
4496,2026-02-16,25423.599609,25697.000000,25372.699219,25682.750000,275800,0.008309,-0.013023,0.006757,-0.009172,0.996614,0.997166,1.006737,0.141515,0.138191
4497,2026-02-17,25637.949219,25764.400391,25570.300781,25725.400391,344100,0.001661,0.008309,0.002623,0.025476,0.999897,0.998830,1.008133,0.140728,0.141515
4498,2026-02-18,25752.650391,25828.050781,25645.150391,25819.349609,310200,0.003652,0.001661,0.000721,0.001883,1.004599,1.002309,1.010652,0.130728,0.140728
4499,2026-02-19,25873.349609,25885.300781,25388.750000,25454.349609,0,-0.014137,0.003652,-0.005650,-0.005168,0.993124,0.988863,0.995786,0.141139,0.130728


In [6]:
features = [
    "ret_lag_1",
    "ret_lag_5",
    "ret_lag_10",
    "ma_ratio_5",
    "ma_ratio_10",
    "ma_ratio_20",
    "vol_cc",
    "vol_cc_lag_1"
]

In [7]:
split = int(len(df) * 0.8)

train = df.iloc[:split]
test = df.iloc[split:]

X_train = train[features]
y_train = train["return"]

X_test = test[features]
y_test = test["return"]

# **Base Quantile Model**
**Using Gradient Boosting**

In [8]:
from sklearn.ensemble import GradientBoostingRegressor

models = {}

for q in [0.1, 0.5, 0.9]:

    model = GradientBoostingRegressor(
        loss="quantile",
        alpha=q,
        n_estimators=200,
        max_depth=3
    )

    model.fit(X_train, y_train)

    models[q] = model

In [9]:
q10_baseline = models[0.1].predict(X_test)
q50_baseline = models[0.5].predict(X_test)
q90_baseline = models[0.9].predict(X_test)

In [10]:
results = test.copy()
results["q10_baseline"] = q10_baseline
results["q50_baseline"] = q50_baseline
results["q90_baseline"] = q90_baseline

results.head()

,Date,Open,High,Low,Close,Volume,return,ret_lag_1,ret_lag_5,ret_lag_10,ma_ratio_5,ma_ratio_10,ma_ratio_20,vol_cc,vol_cc_lag_1,q10_baseline,q50_baseline,q90_baseline
3600,2022-06-30,15774.500000,15890.000000,15728.849609,15780.250000,306000,-0.001193,-0.003224,0.009300,-0.021128,0.999245,1.010174,0.994411,0.176167,0.178987,-0.006465,-0.001954,0.005693
3601,2022-07-01,15703.700195,15793.950195,15511.049805,15752.049805,364100,-0.001787,-0.001193,0.009166,-0.004368,0.996793,1.005417,0.995244,0.176191,0.176167,-0.009042,-0.004921,0.002728
3602,2022-07-04,15710.500000,15852.349609,15661.799805,15835.349609,304300,0.005288,-0.001787,0.008459,0.003704,1.002022,1.007613,1.002833,0.178318,0.176191,-0.004478,0.002080,0.008453
3603,2022-07-05,15909.150391,16025.750000,15785.450195,15810.849609,254200,-0.001547,0.005288,0.001146,0.018804,1.000971,1.004954,1.003205,0.176381,0.178318,-0.008911,-0.002235,0.005443
3604,2022-07-06,15818.200195,16011.349609,15800.900391,15989.799805,288400,0.011318,-0.001547,-0.003224,-0.014419,1.009861,1.012618,1.015740,0.182231,0.176381,0.001523,0.008885,0.013922


In [51]:
coverage_baseline = (
    (y_test >= q10_baseline) &
    (y_test <= q90_baseline)
).mean()
coverage_baseline

np.float64(0.05333333333333334)

In [52]:
width_baseline = (q90_baseline - q10_baseline).mean()
width_baseline

np.float64(0.009592835045150804)

# **Temporal Fusion Transformer Probabilistic Forecasting**

In [11]:
import torch
import pytorch_lightning as pl

from pytorch_forecasting import TimeSeriesDataSet
from pytorch_forecasting.models import TemporalFusionTransformer
from pytorch_forecasting.metrics import QuantileLoss

In [12]:
reg = pd.read_csv(
    "../data/processed/hmm_regimes_refit.csv",
    parse_dates=["Date"]
)
reg.head()

,Date,hmm_regime
0,2007-10-16,High
1,2007-10-17,High
2,2007-10-18,High
3,2007-10-19,High
4,2007-10-22,High


In [13]:
df = df.merge(
    reg[["Date", "hmm_regime"]],
    on="Date",
    how="left"
)

In [14]:
df = df.dropna().reset_index(drop=True)

In [15]:
#TFT requires categorical variables
df["hmm_regime"] = df["hmm_regime"].astype("category")

In [16]:
df["hmm_regime"].value_counts()

hmm_regime
Medium    1669
Low       1663
High      1168
Name: count, dtype: int64

In [17]:
#TFT needs time index and series ID
df["time_idx"] = np.arange(len(df))
df["series"] = "nifty"

In [18]:
df.columns

Index(['Date', 'Open', 'High', 'Low', 'Close', 'Volume', 'return', 'ret_lag_1',
       'ret_lag_5', 'ret_lag_10', 'ma_ratio_5', 'ma_ratio_10', 'ma_ratio_20',
       'vol_cc', 'vol_cc_lag_1', 'hmm_regime', 'time_idx', 'series'],
      dtype='object')

In [19]:
split = int(len(df) * 0.8)

train_df = df.iloc[:split]
test_df = df.iloc[split:]

In [20]:
max_encoder_length = 30
max_prediction_length = 1
#Use past 30 days ->predict next day return

In [21]:
known_reals = ["time_idx"]

unknown_reals = [
    "ret_lag_1",
    "ret_lag_5",
    "ret_lag_10",
    "ma_ratio_5",
    "ma_ratio_10",
    "ma_ratio_20",
    "vol_cc",
    "vol_cc_lag_1",
    "return"
]

categoricals = ["hmm_regime"]

In [22]:
training = TimeSeriesDataSet(
    train_df,

    time_idx="time_idx",
    target="return",
    group_ids=["series"],

    max_encoder_length=max_encoder_length,
    max_prediction_length=max_prediction_length,

    time_varying_known_reals=known_reals,
    time_varying_unknown_reals=unknown_reals,

    time_varying_known_categoricals=categoricals,

    target_normalizer=None
)

In [23]:
validation = TimeSeriesDataSet.from_dataset(
    training,
    test_df,
    predict=True,
    stop_randomization=True
)

In [24]:
#dataloaders
train_loader = training.to_dataloader(
    train=True,
    batch_size=64
)

val_loader = validation.to_dataloader(
    train=False,
    batch_size=64
)

In [25]:
len(training), len(validation)

(3570, 1)

In [26]:
from pytorch_forecasting.models import TemporalFusionTransformer
from pytorch_forecasting.metrics import QuantileLoss

In [34]:
#modelin time
tft = TemporalFusionTransformer.from_dataset(
    training,

    learning_rate=0.03,

    hidden_size=16,
    attention_head_size=1,

    dropout=0.1,

    loss=QuantileLoss(quantiles=[0.1, 0.5, 0.9]),

    output_size=3
)

C:\Users\Aryavart\stock-regime-ml\.venv\Lib\site-packages\lightning\pytorch\utilities\parsing.py:213: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
C:\Users\Aryavart\stock-regime-ml\.venv\Lib\site-packages\lightning\pytorch\utilities\parsing.py:213: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.


In [35]:
#trainin
import lightning.pytorch as pl

trainer = pl.Trainer(
    max_epochs=10,
    accelerator="cpu"
)

GPU available: True (cuda), used: False
TPU available: False, using: 0 TPU cores
C:\Users\Aryavart\stock-regime-ml\.venv\Lib\site-packages\lightning\pytorch\trainer\setup.py:175: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


In [36]:
#Actual trainin lol
trainer.fit(
    tft,
    train_loader,
    val_loader
)

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.

   | Name                               | Type                            | Params | Mode  | FLOPs
--------------------------------------------------------------------------------------------------------
0  | loss                               | QuantileLoss                    | 0      | train | 0    
1  | logging_metrics                    | ModuleList                      | 0      | train | 0    
2  | input_embeddings                   | MultiEmbedding                  | 9      | train | 0    
3  | prescalers                         | ModuleDict                      | 160    | train | 0    
4  | static_variable_selection          | VariableSelectionNetwork        | 0      | train | 0    
5  | encoder_variable_selection         | VariableSelectionNetwork        | 6.9 K  | train | 0 

Sanity Checking: |                                                                               | 0/? [00:00<…

C:\Users\Aryavart\stock-regime-ml\.venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.
C:\Users\Aryavart\stock-regime-ml\.venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.


Training: |                                                                                      | 0/? [00:00<…

Validation: |                                                                                    | 0/? [00:00<…

...

Validation: |                                                                                    | 0/? [00:00<…

Validation: |                                                                                    | 0/? [00:00<…

Validation: |                                                                                    | 0/? [00:00<…

Validation: |                                                                                    | 0/? [00:00<…

`Trainer.fit` stopped: `max_epochs=10` reached.


In [39]:
raw_predictions = tft.predict(
    val_loader,
    mode="raw",
    return_x=True
)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
C:\Users\Aryavart\stock-regime-ml\.venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.


In [43]:
preds = raw_predictions.output.prediction.detach().cpu().numpy()
#Quantile Extraction
q10_tft = preds[:, :, 0].flatten()
q50_tft = preds[:, :, 1].flatten()
q90_tft = preds[:, :, 2].flatten()

In [44]:
y_test = test_df["return"].iloc[-len(q10_tft):].values

In [45]:
coverage_tft = ((y_test >= q10_tft) & (y_test <= q90_tft)).mean()

coverage_tft

np.float64(1.0)

In [46]:
width_tft = (q90_tft - q10_tft).mean()
#interval width
width_tft

np.float32(0.037743382)

In [47]:
def pinball_loss(y, q_pred, alpha):

    error = y - q_pred

    return np.mean(np.maximum(alpha * error, (alpha - 1) * error))

In [48]:
pinball_q10 = pinball_loss(y_test, q10_tft, 0.1)
pinball_q50 = pinball_loss(y_test, q50_tft, 0.5)
pinball_q90 = pinball_loss(y_test, q90_tft, 0.9)

# **Comparison With Gradient Boost**

In [53]:
comparison = {
    "Model": ["Gradient Boost", "TFT"],
    "Coverage": [coverage_baseline, coverage_tft],
    "Interval Width": [width_baseline, width_tft]
}
comparison

{'Model': ['Gradient Boost', 'TFT'],
 'Coverage': [np.float64(0.05333333333333334), np.float64(1.0)],
 'Interval Width': [np.float64(0.009592835045150804), np.float32(0.037743382)]}